# 📘 Modèle Pyomo généré automatiquement

## 📦 Imports

In [ ]:
from pyomo.environ import *
from pyomo.opt import SolverFactory
import pandas as pd

## 🔹 Model

In [ ]:
from pyomo.environ import *

model = ConcreteModel()

## 🔹 Sets

In [ ]:
model.PRODUITS = Set(initialize=[1, 2, 3])
model.MACHINES = Set(initialize=[1, 2, 3])
model.PRODUCTION = Set(dimen=2, initialize=[(i0,i1) for i0 in model.MACHINES for i1 in model.PRODUITS])

## 🔹 Parameters

In [ ]:
model.PROFIT = Param(model.PRODUITS, initialize={1: 50.0, 2: 20.0, 3: 25.0}, within=NonNegativeReals)
model.DEMANDE = Param(model.PRODUITS, initialize={1: 1000.0, 2: 1000.0, 3: 20.0}, within=NonNegativeReals)
model.CAP = Param(model.MACHINES, initialize={1: 500.0, 2: 350.0, 3: 150.0}, within=NonNegativeReals)
model.HRPROD = Param(model.MACHINES, model.PRODUITS, initialize={(1, 1): 9.0, (1, 2): 5.0, (1, 3): 3.0, (2, 1): 3.0, (2, 2): 4.0, (2, 3): 0.0, (3, 1): 5.0, (3, 2): 0.0, (3, 3): 2.0}, within=NonNegativeReals)

## 🔹 Variables

In [ ]:
model.X = Var(model.PRODUITS, domain=NonNegativeIntegers)

## 🔹 Constraints

In [ ]:
model.c_for_0 = ConstraintList()
for p in model.PRODUITS:
    model.c_for_0.add(model.X[p] <= model.DEMANDE[p])
model.c_for_1 = ConstraintList()
for m in model.MACHINES:
    model.c_for_1.add(sum(model.HRPROD[m, p] * model.X[p] for p in model.PRODUITS) <= model.CAP[m])
# @BIN/@GIN directive already handled in variable declarations

## 🔹 Objective

In [ ]:
model.obj = Objective(expr=sum(model.PROFIT[p] * model.X[p] for p in model.PRODUITS), sense=maximize)

## ⚙️ Résolution du modèle

In [ ]:
solver = SolverFactory('highs')
result = solver.solve(model, tee=True)

print('✅ Solver status:', result.solver.status)
print('✅ Termination condition:', result.solver.termination_condition)

## 🎯 Valeur de la fonction objective

In [ ]:
for obj in model.component_objects(Objective, active=True):
    print(f'Objectif: {obj.name}')
    print(f'Valeur optimale: {obj():.4f}')
    print(f'Sens: {"Minimisation" if obj.sense == minimize else "Maximisation"}')

## 📊 Valeurs optimales des variables

In [ ]:
# Extraction des résultats dans un DataFrame
results_data = []
for v in model.component_objects(Var, active=True):
    for index in v:
        results_data.append({
            'Variable': v.name,
            'Index': str(index) if index != None else '-',
            'Valeur': v[index].value
        })

df_results = pd.DataFrame(results_data)
# Filtrer les valeurs non-nulles pour plus de clarté
df_results = df_results[df_results['Valeur'].notna()]
df_results = df_results[df_results['Valeur'] != 0]
df_results.style.format({'Valeur': '{:.4f}'}).set_caption('Variables de décision optimales')